In [ ]:
## 🤖 03: Predictive Modeling and Feature Importance

# --- Setup ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, RocCurveDisplay
from scipy.stats import uniform, randint # For RandomizedSearchCV distributions

# --- Configuration ---
PROCESSED_DATA_PATH = '../data/processed/patient_records_cleaned.csv'
MODEL_SUMMARY_PATH = '../docs/project_summary.md' # Appending insights to the summary document
RANDOM_STATE = 42

In [ ]:
# --- 1. Load Data and Define Features/Target ---
df = pd.read_csv(PROCESSED_DATA_PATH)
# Drop identifiers that are not predictive features
X = df.drop(columns=['Encounter_ID', 'Patient_ID', 'Readmitted_30days'])
y = df['Readmitted_30days']

# Define feature types for preprocessing based on data cleaning
numerical_features = [
    'Length_of_Stay', 'Num_Prior_Admissions', 'Num_Medications', 
    'Num_Lab_Procedures', 'Num_Procedures', 'Comorbidity_Count'
]

# Categorical features need One-Hot Encoding
categorical_features = [
    'Age_Group', 'Gender', 'Race', 'Diagnosis_Category', 'LOS_Bucket'
    # discharge_disposition_id, admission_type_id, etc. would be included if we had kept the lookup table
]

# Binary feature already encoded as 0/1, keep it as 'remainder'
binary_features = ['Medication_Change']

In [ ]:
# --- 2. Train/Validation Split ---
# Stratify to ensure the positive class (Readmitted_30days=1) is represented equally in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train/Test split complete. Train shape: {X_train.shape}")

In [ ]:
# --- 3. Preprocessing Pipeline Definition ---
# Use ColumnTransformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features), # Scale continuous variables
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features) # OHE for categorical
    ],
    remainder='passthrough' # Keep binary features as they are
)

In [ ]:
# --- 4. Model Training and Evaluation ---
metrics_results = [] # To store and compare model results

## 4.1. Baseline Model: Logistic Regression
print("\n--- Training Logistic Regression (Baseline) ---")
logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, solver='liblinear'))
])

logreg_pipeline.fit(X_train, y_train)
y_pred_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]

# Evaluation
lr_roc_auc = roc_auc_score(y_test, y_pred_proba_lr)
lr_pr_auc = average_precision_score(y_test, y_pred_proba_lr)
metrics_results.append({
    'Model': 'Logistic Regression (Baseline)',
    'ROC-AUC': lr_roc_auc,
    'PR-AUC': lr_pr_auc
})
print(f"LR ROC-AUC: {lr_roc_auc:.4f}, PR-AUC: {lr_pr_auc:.4f}")

In [ ]:
## 4.2. Main Model: LightGBM (Untuned)
print("\n--- Training LightGBM (Untuned) ---")
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('classifier', LGBMClassifier(random_state=RANDOM_STATE, n_estimators=100, metric='auc', verbose=-1))
])

lgbm_pipeline.fit(X_train, y_train)
y_pred_proba_lgbm = lgbm_pipeline.predict_proba(X_test)[:, 1]

# Evaluation
lgbm_roc_auc = roc_auc_score(y_test, y_pred_proba_lgbm)
lgbm_pr_auc = average_precision_score(y_test, y_pred_proba_lgbm)
metrics_results.append({
    'Model': 'LightGBM (Untuned)',
    'ROC-AUC': lgbm_roc_auc,
    'PR-AUC': lgbm_pr_auc
})
print(f"LGBM ROC-AUC: {lgbm_roc_auc:.4f}, PR-AUC: {lgbm_pr_auc:.4f}")

In [ ]:
# --- 5. Hyperparameter Tuning (LightGBM) ---
print("\n--- Hyperparameter Tuning (Randomized Search) ---")

# Define parameter search space
param_dist = {
    'classifier__learning_rate': uniform(0.01, 0.2), # between 0.01 and 0.21
    'classifier__num_leaves': randint(20, 50),
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': [-1, 6, 10], # -1 means no limit
    'classifier__min_child_samples': randint(10, 30),
}

# Use the untuned pipeline for the search
random_search = RandomizedSearchCV(
    lgbm_pipeline, 
    param_distributions=param_dist, 
    n_iter=10, # 10 iterations for a quick, representative search
    scoring='roc_auc', 
    cv=3, 
    verbose=0, 
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
best_lgbm = random_search.best_estimator_

# Evaluation of best model
y_pred_proba_best = best_lgbm.predict_proba(X_test)[:, 1]
best_lgbm_roc_auc = roc_auc_score(y_test, y_pred_proba_best)
best_lgbm_pr_auc = average_precision_score(y_test, y_pred_proba_best)

metrics_results.append({
    'Model': 'LightGBM (Tuned)',
    'ROC-AUC': best_lgbm_roc_auc,
    'PR-AUC': best_lgbm_pr_auc
})

print(f"Best LGBM ROC-AUC: {best_lgbm_roc_auc:.4f}, PR-AUC: {best_lgbm_pr_auc:.4f}")
print(f"Best parameters: {random_search.best_params_}")

In [ ]:
# --- 6. Feature Importance and Model Insights ---

# Retrieve feature names after one-hot encoding
ohe_feature_names = best_lgbm['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([numerical_features, ohe_feature_names, binary_features])

# Extract importance from the trained LGBM model
importance = best_lgbm['classifier'].feature_importances_
importance_df = pd.DataFrame({
    'Feature': all_feature_names, 
    'Importance': importance
}).sort_values('Importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(20), palette="viridis")
plt.title('LightGBM Top 20 Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(VISUALS_DIR, 'lgbm_feature_importance.png'))
plt.close()

In [ ]:
# --- 7. Final Model Card and Summary ---
# The LightGBM (Tuned) is the final chosen model.

# Save key metrics to the project summary file
with open(MODEL_SUMMARY_PATH, 'a') as f:
    f.write("\n\n---\n## 🤖 Predictive Model Summary\n")
    f.write(f"**Final Model:** LightGBM (Tuned)\n")
    f.write(f"**ROC-AUC (Test Set):** {best_lgbm_roc_auc:.4f}\n")
    f.write(f"**PR-AUC (Test Set):** {best_lgbm_pr_auc:.4f}\n\n")
    f.write("### Top 5 Predictive Features:\n")
    for _, row in importance_df.head(5).iterrows():
        f.write(f"- {row['Feature']} (Score: {row['Importance']})\n")

print("\nModeling complete. Results and feature importance plot saved.")